## 1. 安裝 Ollama 與 ngrok

In [ ]:
'''
Load the tools needed to install Ollama and ngrok.
載入 Ollama 與 ngrok 安裝所需工具。
'''
import shutil


'''
Run system installation commands and collect errors.
執行系統安裝指令，並保留錯誤資訊。
'''
import subprocess


'''
Install Ollama only when it is not already available.
僅在尚未安裝 Ollama 時進行安裝。
'''
if not shutil.which("ollama"):

    '''
    Ensure the compression utility required by the official installer exists.
    確保已安裝官方安裝腳本所需的 zstd 解壓縮工具。
    '''
    if not shutil.which("zstd"):

        '''
        Refresh the system package index before installing a missing dependency.
        安裝缺少的相依工具前，先更新套件索引。
        '''
        subprocess.run(["apt-get", "update", "-qq"], check=True)


        '''
        Install the zstd command-line tool.
        安裝 zstd 命令列工具。
        '''
        subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)


    '''
    Run the official Ollama installer and capture its diagnostics.
    執行 Ollama 官方安裝腳本，並保留安裝輸出以利診斷。
    '''
    installation = subprocess.run(
        ["bash", "-o", "pipefail", "-c", "curl -fsSL https://ollama.com/install.sh | sh"],
        capture_output=True,
        text=True,
    )


    '''
    Display the real installer error instead of only a process exit code.
    安裝失敗時顯示實際錯誤，而非只有程序結束代碼。
    '''
    if installation.returncode:

        '''
        Report recent installation output for troubleshooting.
        回報最近的安裝輸出，方便排除問題。
        '''
        raise RuntimeError(
            f"Ollama 安裝失敗（Exit Code {installation.returncode}）\n"
            f"STDOUT:\n{installation.stdout[-4000:]}\n"
            f"STDERR:\n{installation.stderr[-4000:]}"
        )


    '''
    Show the installer's completion message.
    顯示安裝腳本完成訊息。
    '''
    print(installation.stdout[-1200:])


'''
Install ngrok only when it is not yet available.
僅在尚未安裝 ngrok 時進行安裝。
'''
if not shutil.which("ngrok"):

    '''
    Install the standalone ngrok executable.
    直接安裝 ngrok 執行檔。
    '''
    subprocess.run(["bash", "-o", "pipefail", "-c", "curl -fsSL https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz | tar -xz -C /usr/local/bin"], check=True)


'''
Check whether the Ollama executable works.
確認 Ollama 執行檔可以正常執行。
'''
subprocess.run(["ollama", "--version"], check=True)


'''
Check whether the ngrok executable works.
確認 ngrok 執行檔可以正常執行。
'''
subprocess.run(["ngrok", "version"], check=True)


'''
Confirm that the installation stage has completed.
顯示安裝階段完成訊息。
'''
print("1. 軟體安裝完成")


## 2. 啟動 Ollama 服務

In [ ]:
'''
Access environment variables for the Ollama server.
讀取 Ollama 服務所需的環境變數。
'''
import os


'''
Wait for the Ollama service to become ready.
等待 Ollama 服務啟動完成。
'''
import time


'''
Prepare API requests without extra Python packages.
使用標準函式庫建立 API 請求。
'''
import urllib.request


'''
Process Ollama API JSON responses.
處理 Ollama API 的 JSON 回覆。
'''
import json


'''
Read service logs when startup fails.
啟動失敗時讀取服務紀錄。
'''
from pathlib import Path


'''
Share the local API address with later cells.
設定後續 Cell 共用的本機 API 位址。
'''
LOCAL_API = "http://127.0.0.1:11434"


'''
Keep the original context length setting.
保留原先的 Context 長度設定。
'''
CONTEXT_LENGTH = 65536


'''
Configure Ollama before starting its service.
啟動服務前設定 Ollama 參數。
'''
ollama_env = {**os.environ, "OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_CONTEXT_LENGTH": str(CONTEXT_LENGTH), "OLLAMA_KEEP_ALIVE": "-1", "OLLAMA_ORIGINS": "*"}


'''
Restart Ollama so the settings take effect.
重新啟動 Ollama，確保設定生效。
'''
subprocess.run(["pkill", "-x", "ollama"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


'''
Allow the old server to release its port.
讓舊服務釋放連接埠。
'''
time.sleep(1)


'''
Start Ollama in the background and preserve its log.
在背景啟動 Ollama 並保留紀錄。
'''
with open("/tmp/ollama.log", "w") as log:

    '''
    Launch the configured server.
    啟動已設定參數的服務。
    '''
    subprocess.Popen(["ollama", "serve"], env=ollama_env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)


'''
Check whether the local API is ready.
確認本機 API 是否完成啟動。
'''
for _ in range(30):

    '''
    Retry until the server responds.
 持續嘗試連接服務。
    '''
    try:

        '''
        Request Ollama's model list endpoint.
        請求 Ollama 模型清單端點。
        '''
        with urllib.request.urlopen(LOCAL_API + "/api/tags", timeout=2):

            '''
            Exit once the service becomes available.
            服務就緒後結束檢查。
            '''
            break

    except OSError:

        '''
        Wait briefly before retrying.
        稍候再重新嘗試。
        '''
        time.sleep(1)

else:

    '''
    Show logs when the service does not start.
    服務啟動失敗時顯示紀錄。
    '''
    raise RuntimeError(Path("/tmp/ollama.log").read_text(errors="replace")[-3000:])


'''
Confirm that the local Ollama API is ready.
顯示本機 Ollama API 已啟動。
'''
print(f"2. Ollama 已啟動：{LOCAL_API}｜Context：{CONTEXT_LENGTH}")

## 3. 下載模型

In [ ]:
'''
Choose the model to download.
設定要下載的模型名稱。
'''
MODEL_TO_PULL = "nemotron-3.5-bolt:e4b"


'''
Download the chosen model from Ollama.
透過 Ollama 下載指定模型。
'''
subprocess.run(["ollama", "pull", MODEL_TO_PULL], check=True)


'''
Confirm the model download finished.
顯示模型下載完成訊息。
'''
print(f"3. 模型已下載：{MODEL_TO_PULL}")

## 4. 載入模型

In [ ]:
'''
Check which model was previously active.
查看先前載入的模型。
'''
previous_model = globals().get("ACTIVE_MODEL")


'''
Unload the previous model when switching models.
切換模型時先卸載舊模型。
'''
if previous_model and previous_model != MODEL_TO_PULL:

    '''
    Release memory used by the old model.
    釋放舊模型使用的記憶體。
    '''
    subprocess.run(["ollama", "stop", previous_model], check=True)


'''
Prepare a request that preloads the new model.
建立預先載入新模型的 API 請求。
'''
load_request = urllib.request.Request(
    LOCAL_API + "/api/generate",
    data=json.dumps({"model": MODEL_TO_PULL, "keep_alive": -1, "stream": False}).encode(),
    headers={"Content-Type": "application/json"},
)


'''
Load the model before updating the active selection.
成功載入模型後才更新目前使用的模型。
'''
with urllib.request.urlopen(load_request, timeout=900) as response:

    '''
    Verify that Ollama returned valid JSON.
    確認 Ollama 正常回傳 JSON。
    '''
    json.load(response)


'''
Share the active model with all later cells.
讓後續 Cell 共用目前使用的模型。
'''
ACTIVE_MODEL = MODEL_TO_PULL


'''
Display the active model and its memory usage.
顯示目前模型及記憶體載入狀態。
'''
print(f"4. 目前使用模型：{ACTIVE_MODEL}")


'''
Check whether Ollama has loaded the model.
查看 Ollama 已載入的模型。
'''
subprocess.run(["ollama", "ps"], check=True)

## 5. 測試模型

In [ ]:
'''
Create a local chat request using the active model.
使用目前模型建立本機對話請求。
'''
chat_request = urllib.request.Request(
    LOCAL_API + "/api/chat",
    data=json.dumps({"model": ACTIVE_MODEL, "messages": [{"role": "user", "content": "請用繁體中文簡短打招呼。"}], "stream": False, "keep_alive": -1}).encode(),
    headers={"Content-Type": "application/json"},
)


'''
Send the local request and read the response.
發送本機請求並讀取模型回覆。
'''
with urllib.request.urlopen(chat_request, timeout=900) as response:

    '''
    Parse the model's reply.
    解析模型回覆內容。
    '''
    result = json.load(response)


'''
Display the response for the student.
顯示模型生成的內容。
'''
print(f"5. {ACTIVE_MODEL}：{result['message'].get('content') or result['message'].get('thinking') or '（沒有文字回覆）'}")

## 6. 查看模型狀態

In [ ]:
'''
Display the selected model.
顯示目前使用的模型。
'''
print("目前使用模型：", globals().get("ACTIVE_MODEL", "尚未載入"))


'''
List all downloaded models.
列出已下載的模型。
'''
subprocess.run(["ollama", "list"], check=True)


'''
List models currently loaded in memory.
列出目前載入記憶體的模型。
'''
subprocess.run(["ollama", "ps"], check=True)


'''
Confirm the model status check finished.
顯示模型狀態檢查完成訊息。
'''
print("6. 模型狀態檢查完成")

## 7. 設定 ngrok Authtoken

In [ ]:
'''
Read the ngrok account credential privately.
以隱藏輸入方式取得 ngrok 帳戶憑證。
'''
from getpass import getpass


'''
Ask for the account Authtoken required to run ngrok.
輸入執行 ngrok 所需的帳戶 Authtoken。
'''
ngrok_authtoken = getpass("請輸入 ngrok Authtoken：").strip()


'''
Reject an empty ngrok account credential.
避免設定空白的 ngrok 帳戶憑證。
'''
if not ngrok_authtoken:

    '''
    Explain why ngrok cannot be configured.
    提示必須提供 ngrok 帳戶憑證。
    '''
    raise ValueError("尚未輸入 ngrok Authtoken")


'''
Save the ngrok account credential without printing it.
儲存 ngrok 帳戶憑證，不顯示於輸出。
'''
subprocess.run(["ngrok", "config", "add-authtoken", ngrok_authtoken], check=True, stdout=subprocess.DEVNULL)


'''
Avoid retaining the credential in a Notebook variable.
避免在 Notebook 變數中保留憑證。
'''
del ngrok_authtoken


'''
Confirm the ngrok configuration is complete.
顯示 ngrok 設定完成訊息。
'''
print("7. ngrok Authtoken 設定完成")

## 8. 啟動 ngrok 公開 Ollama API

In [ ]:
'''
Confirm that Ollama is available before exposing it.
建立通道前確認 Ollama 已啟動。
'''
with urllib.request.urlopen(LOCAL_API + "/api/tags", timeout=3):

    '''
    Complete the availability check.
    完成本機服務連線檢查。
    '''
    pass


'''
Close an earlier ngrok tunnel before starting a new one.
建立新通道前先關閉舊的 ngrok 通道。
'''
subprocess.run(["pkill", "-x", "ngrok"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


'''
Allow ngrok to release its local inspection port.
讓 ngrok 釋放本機監控連接埠。
'''
time.sleep(1)


'''
Remember where ngrok startup messages are written.
設定 ngrok 啟動紀錄的檔案位置。
'''
NGROK_LOG = Path("/tmp/ngrok.log")


'''
Expose Ollama directly and rewrite its Host header.
直接對外開放 Ollama，並修正轉送的 Host 標頭。
'''
with NGROK_LOG.open("w") as log:

    '''
    Run ngrok in the background without API authentication.
    在背景啟動 ngrok，不設定 API 存取驗證。
    '''
    subprocess.Popen(["ngrok", "http", "11434", "--host-header=localhost:11434"], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)


'''
Clear the previous public URL when restarting ngrok.
重新啟動 ngrok 時清除舊網址。
'''
NGROK_URL = ""


'''
Wait for the public HTTPS URL to become available.
等待 ngrok 產生公開 HTTPS 網址。
'''
for _ in range(30):

    '''
    Read the tunnel information from ngrok's local API.
    從 ngrok 本機 API 取得通道資訊。
    '''
    try:

        '''
        Retrieve the tunnels created by ngrok.
        取得 ngrok 建立的通道清單。
        '''
        with urllib.request.urlopen("http://127.0.0.1:4040/api/tunnels", timeout=2) as response:

            '''
            Parse the list of available tunnels.
            解析目前的通道清單。
            '''
            tunnels = json.load(response).get("tunnels", [])

        '''
        Select the public HTTPS tunnel address.
        選擇公開的 HTTPS 通道網址。
        '''
        NGROK_URL = next((item["public_url"] for item in tunnels if item.get("public_url", "").startswith("https://")), "")

        '''
        Stop polling when a public address is ready.
        已取得公開網址時停止查詢。
        '''
        if NGROK_URL:

            '''
            Leave the polling loop once connected.
            已成功連線時結束等待。
            '''
            break

    except (OSError, ValueError, KeyError):

        '''
        Ignore temporary startup errors.
        略過啟動期間的暫時性錯誤。
        '''
        pass

    '''
    Retry after a short delay.
    稍候再嘗試取得網址。
    '''
    time.sleep(1)


'''
Show the ngrok log if no public URL was created.
無法建立公開網址時顯示 ngrok 紀錄。
'''
if not NGROK_URL:

    '''
    Report the most recent tunnel error.
    顯示最近的通道錯誤訊息。
    '''
    raise RuntimeError("ngrok 啟動失敗：\n" + NGROK_LOG.read_text(errors="replace")[-3000:])


'''
Display the public URL used for direct Ollama API calls.
顯示直接呼叫 Ollama API 的公開網址。
'''
print(f"8. ngrok 已啟動\nOllama API：{NGROK_URL}\nChat API：{NGROK_URL}/api/chat")

## 9. 測試遠端 Ollama API

In [ ]:
'''
Create a remote chat request without access-token headers.
建立不需要 API 存取權杖的遠端對話請求。
'''
remote_request = urllib.request.Request(
    NGROK_URL + "/api/chat",
    data=json.dumps({"model": ACTIVE_MODEL, "messages": [{"role": "user", "content": "請用繁體中文介紹你自己。"}], "stream": False}).encode(),
    headers={"Content-Type": "application/json", "ngrok-skip-browser-warning": "true"},
)


'''
Send the request directly to ngrok's public URL.
直接對 ngrok 公開網址發送請求。
'''
with urllib.request.urlopen(remote_request, timeout=900) as response:

    '''
    Read the reply from the remote Ollama API.
    讀取遠端 Ollama API 的回覆。
    '''
    remote_result = json.load(response)


'''
Show that the API call succeeded and print the reply.
顯示遠端 API 呼叫成功與模型回覆。
'''
print(f"9. 遠端 API 測試成功：{NGROK_URL}/api/chat\n模型：{ACTIVE_MODEL}\n回覆：{remote_result['message'].get('content') or remote_result['message'].get('thinking') or '（沒有文字回覆）'}")

## 10. 卸載模型或停止服務

In [ ]:
'''
Choose whether to unload the active model.
設定是否卸載目前使用的模型。
'''
UNLOAD_MODEL = False


'''
Choose whether to close the public tunnel.
設定是否關閉公開通道。
'''
STOP_NGROK = False


'''
Choose whether to shut down Ollama.
設定是否關閉 Ollama 服務。
'''
STOP_OLLAMA = False


'''
Unload the active model when requested.
有指定時才卸載目前使用的模型。
'''
if UNLOAD_MODEL:

    '''
    Release memory used by the active model.
    釋放目前模型占用的記憶體。
    '''
    subprocess.run(["ollama", "stop", ACTIVE_MODEL], check=True)


'''
Stop the public tunnel when requested.
有指定時才停止公開通道。
'''
if STOP_NGROK:

    '''
    End the ngrok process.
    結束 ngrok 程序。
    '''
    subprocess.run(["pkill", "-x", "ngrok"], check=False)


'''
Stop the Ollama server when requested.
有指定時才停止 Ollama 服務。
'''
if STOP_OLLAMA:

    '''
    End the Ollama service process.
    結束 Ollama 服務程序。
    '''
    subprocess.run(["pkill", "-x", "ollama"], check=False)


'''
Confirm the shutdown controls have been checked.
顯示停止服務設定已執行。
'''
print("10. 停止服務設定已執行")